# TextGraphicalizer: Aesop fables

Run the Laya-backed `TextGraphicalizer` on stories loaded from the cached
Project Gutenberg Aesop corpus using the checked-in high-level WordNet
ontology. The estimator owns the parameterized graph display function, so
rendering can be reused outside this notebook.

> The first model load downloads the pinned Laya checkpoint into the Hugging Face cache.
> The first corpus load downloads and caches the cleaned Aesop stories.


## One-time setup

Use the Python environment selected for this project. From the repository root, run these commands once in a terminal:

```bash
python -m pip install -e ".[notebook]"
python -m ipykernel install --user --name textgraphicalizer --display-name "TextGraphicalizer"
```

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

assert (3, 10) <= sys.version_info[:2] <= (3, 12), (
    f"Use a Python 3.10–3.12 kernel; current interpreter is {sys.version}"
)
print(f"Using Python: {sys.executable}")

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "ontology.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from textgraphicalizer import TextGraphicalizer, load_aesop_fables

plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 15, "font.size": 10})
ontology_path = ROOT / "ontology.yaml"
stopwords_path = ROOT / "stopwords.yaml"
ontology_path

Using Python: /Users/f.costa/.venvs/py312/bin/python


PosixPath('/Users/f.costa/Code/TextGraphicalizer/ontology.yaml')

In [ ]:
extractor = TextGraphicalizer()

## Aesop stories

In [ ]:
stories = load_aesop_fables()
selected_stories = stories[0:1]
documents = [
    (story.split("\n\n", 1)[0], story)
    for story in selected_stories
]
print(f"Loaded {len(stories)} stories; graphicalizing {len(documents)}.")
for index, (title, document) in enumerate(documents, 1):
    print(f"\n--- Story {index}: {title} ---")
    print(document)

In [ ]:
extractor.ontology = ontology_path
extractor.stopwords_path = stopwords_path

In [ ]:
extractor.node_threshold = 0.3
extractor.edge_threshold = 0.3
extractor.connected = True
extractor.max_node_degree = 3
extractor.use_milp = True
extractor.use_llm = True

In [ ]:

graphs = []
for title, document in documents:
    print(f"Document:\n{document}")

    graph = extractor.transform(document)
    graphs.append((title, document, graph))
    
    print(f"{title}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
for title, document, graph in graphs:
    extractor.display(
        graph,
        title=title,
        document=document,
        max_char=150,
    )


In [ ]:
for title, document, graph in graphs:
    display(extractor.display_d3(
        graph,
        title=title,
        document=document,
        max_char=100,
    ))


## Inspect one graph as ordinary NetworkX data

In [ ]:
title, document, graph = graphs[0]
print("Story:", title)
print("Nodes:")
display(list(graph.nodes(data=True)))
print("Edges:")
display(list(graph.edges(data=True)))
print("Graph metadata:")
display(graph.graph)
